Hotel Price Prediction – Model Training

1. Načtení dat

In [ ]:
import pandas as pd
import numpy as np
import json, pickle, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

df = pd.read_csv("dataset.csv")
print(f"Řádků: {len(df)}")
df.head()

2. Čištění

In [ ]:
df["city"]      = df["city"].str.capitalize()
df["breakfast"] = df["breakfast"].astype(int)
df["stars"]     = df["stars"] / 2

if "event_count" not in df.columns:
    df["event_count"] = 0
df["event_count"] = df["event_count"].fillna(0)

df = df.dropna(subset=["price", "rating", "stars", "distance_km", "review_count"])
print(f"Po čištění: {len(df)} řádků")

3. Features

In [ ]:
df["checkin_dt"]      = pd.to_datetime(df["checkin"])
df["month"]           = df["checkin_dt"].dt.month
df["week_of_year"]    = df["checkin_dt"].dt.isocalendar().week.astype(int)
df["log_review_count"] = np.log1p(df["review_count"])
df["log_distance"]    = np.log1p(df["distance_km"])
df["log_price"]       = np.log1p(df["price"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["price"],     bins=50, color="steelblue"); axes[0].set_title("Cena (raw)")
axes[1].hist(df["log_price"], bins=50, color="seagreen");  axes[1].set_title("Cena (log)")
plt.tight_layout(); plt.show()

4. Train / Test Split

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
print(f"Train: {len(df_train)}  |  Test: {len(df_test)}")

5. City Encoding & Event Features

In [ ]:
city_mean   = df_train.groupby("city")["log_price"].mean()
global_mean = df_train["log_price"].mean()
event_mean  = df_train["event_count"].mean()

df_train = df_train.copy()
df_test  = df_test.copy()

df_train["city_encoded"] = df_train["city"].map(city_mean)
df_test["city_encoded"]  = df_test["city"].map(city_mean).fillna(global_mean)

df_train["high_event"] = (df_train["event_count"] > event_mean).astype(int)
df_test["high_event"]  = (df_test["event_count"] > event_mean).astype(int)

print(city_mean.sort_values(ascending=False).round(3))

In [ ]:
FEATURES = [
    "rating", "stars", "breakfast", "log_distance", "log_review_count",
    "month", "day_of_week", "is_weekend", "stay_length", "week_of_year",
    "event_count", "high_event", "city_encoded",
]
TARGET = "log_price"

X_train, y_train = df_train[FEATURES], df_train[TARGET]
X_test,  y_test  = df_test[FEATURES],  df_test[TARGET]
print(f"Features: {FEATURES}")

6. Trénování modelu

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
)
xgb_model.fit(X_train, y_train)

rf_model = RandomForestRegressor(
    n_estimators=200, max_depth=12, random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)


8. Vyhodnocení

In [ ]:
def evaluate(name, y_true_log, y_pred_log):
    y_true, y_pred = np.expm1(y_true_log), np.expm1(y_pred_log)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"{name}  |  MAE: {mae:.0f} Kč  |  RMSE: {rmse:.0f} Kč  |  R²: {r2:.4f}")
    return {"mae": mae, "rmse": rmse, "r2": r2}

metrics_xgb = evaluate("XGBoost – test",       y_test, xgb_model.predict(X_test))
metrics_rf  = evaluate("Random Forest – test",  y_test, rf_model.predict(X_test))

y_pred_real = np.expm1(xgb_model.predict(X_test))
y_true_real = np.expm1(y_test)
plt.figure(figsize=(6, 6))
plt.scatter(y_true_real, y_pred_real, alpha=0.3, s=8, color="steelblue")
m = max(y_true_real.max(), y_pred_real.max())
plt.plot([0, m], [0, m], "r--", linewidth=1)
plt.xlabel("Skutečná cena (Kč)"); plt.ylabel("Predikovaná cena (Kč)")
plt.title("Skutečná vs predikovaná cena")
plt.tight_layout(); plt.show()

9. Feature Importance

In [ ]:
fi = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values()
fi.plot(kind="barh", figsize=(8, 5), color="steelblue", edgecolor="white")
plt.title("Feature Importance – XGBoost")
plt.tight_layout(); plt.show()

10. Uložení

In [ ]:
xgb_model.save_model("model_xgb.json")

with open("model_rf.pkl", "wb") as f:
    pickle.dump(rf_model, f)

with open("city_mean.json", "w") as f:
    json.dump(city_mean.to_dict(), f, ensure_ascii=False, indent=2)

results = {
    "xgboost":               metrics_xgb,
    "random_forest":         metrics_rf,
    "features":              FEATURES,
    "train_size":            len(df_train),
    "test_size":             len(df_test),
    "event_mean_threshold":  float(event_mean),
    "global_price_mean_log": float(global_mean),
}
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)
